# [SQL 실습 #07] MySQL 데이터 삭제하기
## DELETE와 TRUNCATE 차이

> **학생용 실습 노트북 - TODO + 자동점검형**

지난 시간에는 `UPDATE`로 데이터를 수정했습니다.  
이번 시간에는 **데이터를 삭제하는 방법**을 연습합니다.

### 오늘의 학습 목표
1. `DELETE` 기본 문법을 이해한다.
2. 삭제 전에 `SELECT`로 대상을 먼저 확인한다.
3. `WHERE`로 특정 행만 안전하게 삭제한다.
4. 이름보다 `id` 같은 고유값이 안전한 이유를 이해한다.
5. 조건에 맞는 여러 행을 삭제할 수 있다.
6. `WHERE` 없는 `DELETE`의 위험성을 설명할 수 있다.
7. `DELETE → AUTO_INCREMENT`의 관계를 확인한다.
8. `TRUNCATE TABLE`의 특징을 이해한다.
9. `DELETE`와 `TRUNCATE`의 차이를 설명할 수 있다.
10. `DROP TABLE`과 데이터 삭제를 혼동하지 않는다.

> **핵심:**  
> `DELETE`는 조건에 맞는 데이터를 삭제하고,  
> `TRUNCATE`는 테이블 안의 데이터를 한 번에 비웁니다.

### 자동점검 방식
각 TODO는 단순히 결과만 비교하지 않습니다.

- SQL이 실제로 실행되는지
- 요구한 결과가 맞는지
- 문제에서 요구한 `WHERE`, `id`, `DELETE`, `TRUNCATE` 등의 문법을 사용했는지
- 삭제하지 말아야 할 다른 행이 그대로 남아 있는지

를 함께 확인합니다.

# 1. MySQL 실습 환경 준비

아래 셀은 Colab에 MySQL을 설치하고 실행합니다.  
이미 설치되어 있어도 다시 실행해도 됩니다.

> 이 셀은 수정하지 마세요.

In [ ]:
!sudo apt-get -qq update
!sudo DEBIAN_FRONTEND=noninteractive apt-get -qq install -y mysql-server > /dev/null
!sudo service mysql start
print("✅ MySQL 서버 준비 완료")

# 2. `school.students` 실습 데이터 준비

SQL-7 자료의 시작 데이터 **9개**를 그대로 사용합니다.

| id | name | grade | class_name |
|---:|---|---:|---|
| 1 | 김부장 | 3 | 컴퓨터과 |
| 2 | 혼이 | 1 | 영혼반 |
| 3 | 라즈베리 | 2 | 임베디드반 |
| 4 | 리눅스 | 3 | 서버반 |
| 5 | 파이썬 | 1 | 프로그래밍반 |
| 6 | 김코딩 | 1 | 컴퓨터과 |
| 7 | 데이터왕 | 2 | 데이터베이스반 |
| 8 | 홍길동 | 2 | 컴퓨터과 |
| 9 | 이몽룡 | 3 | 데이터베이스반 |

자료 화면과 맞추기 위해 `created_at`은 `2026-08-01 21:00:00`으로 고정합니다.

> 아래 셀은 수업 시작용 초기화 셀입니다.

In [ ]:
%%bash
sudo mysql <<'SQL'
CREATE DATABASE IF NOT EXISTS school
DEFAULT CHARACTER SET utf8mb4
DEFAULT COLLATE utf8mb4_unicode_ci;

USE school;

DROP TABLE IF EXISTS students;

CREATE TABLE students (
    id INT AUTO_INCREMENT PRIMARY KEY,
    name VARCHAR(50) NOT NULL,
    grade INT,
    class_name VARCHAR(50),
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

INSERT INTO students (name, grade, class_name, created_at)
VALUES
('김부장', 3, '컴퓨터과', '2026-08-01 21:00:00'),
('혼이', 1, '영혼반', '2026-08-01 21:00:00'),
('라즈베리', 2, '임베디드반', '2026-08-01 21:00:00'),
('리눅스', 3, '서버반', '2026-08-01 21:00:00'),
('파이썬', 1, '프로그래밍반', '2026-08-01 21:00:00'),
('김코딩', 1, '컴퓨터과', '2026-08-01 21:00:00'),
('데이터왕', 2, '데이터베이스반', '2026-08-01 21:00:00'),
('홍길동', 2, '컴퓨터과', '2026-08-01 21:00:00'),
('이몽룡', 3, '데이터베이스반', '2026-08-01 21:00:00');
SQL
echo "✅ SQL-7 기준 9개 실습 데이터 준비 완료"

# 3. 안전한 SQL 실행 / 자동점검 도우미

삭제 단원은 실수 한 번으로 데이터 전체가 사라질 수 있으므로 안전장치를 적용합니다.

### 안전 규칙
- 학생용 `DELETE` 실행 함수는 **WHERE가 없는 DELETE를 차단**합니다.
- `WHERE 1=1`, `WHERE TRUE`처럼 사실상 전체 행을 선택하는 조건도 차단합니다.
- `DROP TABLE`은 설명만 하고 학생 실습에서는 실행하지 않습니다.
- `TRUNCATE`는 지정된 TODO에서만 별도 함수로 실행합니다.
- 자동점검은 **결과 + 요구 문법**을 함께 검사합니다.

> 이 셀은 수정하지 마세요.

In [ ]:
import subprocess, textwrap, re

TODO_STATUS = {}
LAST_SQL = ""
LAST_MODE = ""

def _mysql(sql, no_header=False):
    cmd = ["sudo", "mysql", "--batch", "--raw"]
    if no_header:
        cmd.append("-N")
    r = subprocess.run(cmd, input=textwrap.dedent(sql).strip(),
                       text=True, capture_output=True)
    return r.returncode == 0, r.stdout.strip(), r.stderr.strip()

def _real(sql):
    return "\n".join(
        line for line in textwrap.dedent(sql).splitlines()
        if line.strip() and not line.lstrip().startswith("--")
    ).strip()

def _has_todo(sql):
    return "TODO" in _real(sql)

def run_select(sql):
    global LAST_SQL, LAST_MODE
    LAST_SQL, LAST_MODE = sql, "select"
    real = _real(sql)
    if not real:
        print("⚠️ SQL이 비어 있습니다."); return False
    if _has_todo(sql):
        print("⚠️ TODO가 남아 있습니다."); return False
    if re.search(r"\b(DELETE|UPDATE|INSERT|TRUNCATE|DROP|ALTER)\b", real, re.I):
        print("❌ 이 셀은 조회 전용입니다."); return False
    ok,out,err = _mysql(sql)
    if out: print(out)
    if not ok:
        print("❌ MySQL 오류"); print(err); return False
    print("✅ 조회 완료"); return True

def _validate_delete(sql):
    real = _real(sql)
    if not real: return False, "SQL이 비어 있습니다."
    if "TODO" in real: return False, "TODO가 남아 있습니다."
    if len(re.findall(r"\bDELETE\s+FROM\b", real, re.I)) != 1:
        return False, "DELETE FROM 한 문장만 작성하세요."
    if not re.search(r"\bWHERE\b", real, re.I):
        return False, "WHERE가 없습니다. 전체 데이터 삭제 위험 때문에 실행을 차단했습니다."
    if re.search(r"\bWHERE\s+(1\s*=\s*1|TRUE)\b", real, re.I):
        return False, "모든 행과 일치하는 위험한 WHERE 조건은 차단합니다."
    if re.search(r"\b(DROP|TRUNCATE|ALTER|UPDATE|INSERT)\b", real, re.I):
        return False, "DELETE 이외의 변경 명령이 들어 있습니다."
    return True, ""

def run_delete(sql):
    global LAST_SQL, LAST_MODE
    LAST_SQL, LAST_MODE = sql, "delete"
    safe,msg = _validate_delete(sql)
    if not safe:
        print("🚫 안전장치:", msg); return False
    ok,out,err = _mysql(sql)
    if out: print(out)
    if not ok:
        print("❌ MySQL 오류"); print(err); return False
    print("✅ DELETE 실행 완료"); return True

def run_insert(sql):
    global LAST_SQL, LAST_MODE
    LAST_SQL, LAST_MODE = sql, "insert"
    real=_real(sql)
    if not real or _has_todo(sql):
        print("⚠️ TODO가 남아 있거나 SQL이 비어 있습니다."); return False
    if len(re.findall(r"\bINSERT\s+INTO\b", real, re.I)) != 1:
        print("❌ INSERT INTO 한 문장만 작성하세요."); return False
    if re.search(r"\b(DELETE|UPDATE|TRUNCATE|DROP|ALTER)\b", real, re.I):
        print("❌ INSERT 이외의 변경 명령이 있습니다."); return False
    ok,out,err=_mysql(sql)
    if out: print(out)
    if not ok:
        print("❌ MySQL 오류"); print(err); return False
    print("✅ INSERT 실행 완료"); return True

def run_truncate(sql):
    global LAST_SQL, LAST_MODE
    LAST_SQL, LAST_MODE = sql, "truncate"
    real=_real(sql)
    if not real or _has_todo(sql):
        print("⚠️ TODO가 남아 있거나 SQL이 비어 있습니다."); return False
    # USE school; + TRUNCATE TABLE students; 형태만 허용
    truncs = re.findall(r"\bTRUNCATE\s+TABLE\s+students\b", real, re.I)
    if len(truncs) != 1:
        print("❌ 이 실습에서는 TRUNCATE TABLE students 한 문장만 사용하세요."); return False
    if re.search(r"\b(DELETE|UPDATE|DROP|ALTER)\b", real, re.I):
        print("❌ TRUNCATE 이외의 변경 명령이 있습니다."); return False
    ok,out,err=_mysql(sql)
    if out: print(out)
    if not ok:
        print("❌ MySQL 오류"); print(err); return False
    print("✅ TRUNCATE 실행 완료"); return True

def _snapshot():
    ok,out,err=_mysql("""
    SELECT id,name,grade,class_name
    FROM school.students ORDER BY id;
    """, no_header=True)
    if not ok: raise RuntimeError(err)
    return [line.split("\t") for line in out.splitlines() if line.strip()]

STAGES = {
 "start":[['1','김부장','3','컴퓨터과'],['2','혼이','1','영혼반'],['3','라즈베리','2','임베디드반'],['4','리눅스','3','서버반'],['5','파이썬','1','프로그래밍반'],['6','김코딩','1','컴퓨터과'],['7','데이터왕','2','데이터베이스반'],['8','홍길동','2','컴퓨터과'],['9','이몽룡','3','데이터베이스반']],
 "after_hong":[['1','김부장','3','컴퓨터과'],['2','혼이','1','영혼반'],['3','라즈베리','2','임베디드반'],['4','리눅스','3','서버반'],['5','파이썬','1','프로그래밍반'],['6','김코딩','1','컴퓨터과'],['7','데이터왕','2','데이터베이스반'],['9','이몽룡','3','데이터베이스반']],
 "after_id5":[['1','김부장','3','컴퓨터과'],['2','혼이','1','영혼반'],['3','라즈베리','2','임베디드반'],['4','리눅스','3','서버반'],['6','김코딩','1','컴퓨터과'],['7','데이터왕','2','데이터베이스반'],['9','이몽룡','3','데이터베이스반']],
 "after_grade1":[['1','김부장','3','컴퓨터과'],['3','라즈베리','2','임베디드반'],['4','리눅스','3','서버반'],['7','데이터왕','2','데이터베이스반'],['9','이몽룡','3','데이터베이스반']],
 "after_id7":[['1','김부장','3','컴퓨터과'],['3','라즈베리','2','임베디드반'],['4','리눅스','3','서버반'],['9','이몽룡','3','데이터베이스반']],
 "after_new":[['1','김부장','3','컴퓨터과'],['3','라즈베리','2','임베디드반'],['4','리눅스','3','서버반'],['9','이몽룡','3','데이터베이스반'],['10','새학생','1','컴퓨터과']],
 "after_restart":[['1','다시시작','1','SQL반']],
}

def _syntax_ok(sql, required=(), forbidden=()):
    real=_real(sql).upper()
    for pat in required:
        if not re.search(pat, real, re.I):
            return False, f"요구 문법이 빠졌습니다: {pat}"
    for pat in forbidden:
        if re.search(pat, real, re.I):
            return False, f"사용하면 안 되는 문법이 포함되었습니다: {pat}"
    return True, ""

def check_select(todo_id, sql, expected_sql, required=()):
    print(f"🔎 TODO {todo_id} 자동점검")
    if _has_todo(sql):
        TODO_STATUS[todo_id]=False; print("❌ TODO가 남아 있습니다."); return False
    syn,msg=_syntax_ok(sql, required)
    if not syn:
        TODO_STATUS[todo_id]=False; print("❌",msg); return False
    ok1,o1,e1=_mysql(sql); ok2,o2,e2=_mysql(expected_sql)
    if not ok1:
        TODO_STATUS[todo_id]=False; print("❌ 학생 SQL 실행 오류"); print(e1); return False
    passed = [x.rstrip() for x in o1.splitlines() if x.strip()] == [x.rstrip() for x in o2.splitlines() if x.strip()]
    TODO_STATUS[todo_id]=passed
    print("✅ 통과: 조회 결과와 요구 문법이 정확합니다." if passed else "❌ 재확인: 조회 결과가 요구사항과 다릅니다.")
    return passed

def check_stage(todo_id, stage, sql=None, required=()):
    print(f"🔎 TODO {todo_id} 자동점검")
    if sql is not None:
        if _has_todo(sql):
            TODO_STATUS[todo_id]=False; print("❌ TODO가 남아 있습니다."); return False
        syn,msg=_syntax_ok(sql, required)
        if not syn:
            TODO_STATUS[todo_id]=False; print("❌",msg); return False
    actual=_snapshot(); expected=STAGES[stage]
    passed=actual==expected
    TODO_STATUS[todo_id]=passed
    if passed:
        print("✅ 통과: 요구 문법과 테이블 상태가 정확합니다.")
    else:
        print("❌ 재확인: 예상하지 않은 행이 삭제되었거나 데이터 상태가 다릅니다.")
        print("💡 데이터가 꼬였다면 맨 아래 초기화 셀을 실행하세요.")
    return passed

def check_empty(todo_id, sql):
    print(f"🔎 TODO {todo_id} 자동점검")
    syn,msg=_syntax_ok(sql, (r"\bTRUNCATE\s+TABLE\b", r"\bSTUDENTS\b"))
    if not syn:
        TODO_STATUS[todo_id]=False; print("❌",msg); return False
    ok,out,err=_mysql("SELECT COUNT(*) FROM school.students;", no_header=True)
    passed=ok and out.strip()=="0"
    TODO_STATUS[todo_id]=passed
    print("✅ 통과: students 구조는 남아 있고 데이터는 0건입니다." if passed else "❌ students 데이터가 비어 있지 않습니다.")
    return passed

def check_table_exists(todo_id):
    print(f"🔎 TODO {todo_id} 자동점검")
    ok,out,err=_mysql("SHOW TABLES FROM school LIKE 'students';", no_header=True)
    passed=ok and out.strip()=="students"
    TODO_STATUS[todo_id]=passed
    print("✅ 통과: students 테이블 구조가 남아 있습니다." if passed else "❌ students 테이블을 찾을 수 없습니다.")
    return passed

def check_auto_increment(todo_id):
    print(f"🔎 TODO {todo_id} 자동점검")
    snap=_snapshot()
    passed=(snap==STAGES["after_new"])
    TODO_STATUS[todo_id]=passed
    print("✅ 통과: DELETE 후 새 데이터의 id가 10으로 이어졌습니다." if passed else "❌ 새 데이터의 id 또는 현재 테이블 상태를 확인하세요.")
    return passed

def explain_danger(sql):
    real=_real(sql)
    print("검사할 SQL:", real)
    safe,msg=_validate_delete(sql)
    if safe: print("✅ WHERE가 있는 DELETE입니다.")
    else: print("🚫 실행 차단:",msg)

def show_progress(total=18):
    done=sum(TODO_STATUS.get(i) is True for i in range(1,total+1))
    fail=sum(TODO_STATUS.get(i) is False for i in range(1,total+1))
    print("="*55); print("📊 SQL 실습 #07 TODO 진행 현황"); print("="*55)
    for s in range(1,total+1,6):
        print("   ".join(f"{i}:{'✅' if TODO_STATUS.get(i) is True else '❌' if TODO_STATUS.get(i) is False else '⬜'}"
                         for i in range(s,min(s+6,total+1))))
    print("-"*55)
    print(f"통과: {done}/{total} | 재확인: {fail}/{total} | 미점검: {total-done-fail}/{total}")
    print(f"진행률: {done/total*100:.1f}%")

# 4. 사용할 데이터베이스 확인하기

## TODO 1
`school` 데이터베이스를 선택하고 테이블 목록을 확인하세요.

힌트:
```sql
USE 데이터베이스명;
SHOW TABLES;
```

In [ ]:
sql = '''
USE TODO;
SHOW TODO;
'''
_todo_id = 1
run_select(sql)

In [ ]:
# TODO 1 자동점검 - 수정하지 마세요.
check_select(1, sql, "USE school; SHOW TABLES;", (r"\bUSE\s+SCHOOL\b", r"\bSHOW\s+TABLES\b"))

# 5. 실습 데이터 전체 확인하기

삭제하기 전에 현재 데이터를 먼저 확인합니다.

## TODO 2
`students` 테이블의 전체 데이터를 조회하세요.

In [ ]:
sql = '''
USE school;
SELECT TODO FROM TODO;
'''
_todo_id = 2
run_select(sql)

In [ ]:
# TODO 2 자동점검
check_select(2, sql, "USE school; SELECT * FROM students;", (r"\bSELECT\b", r"\bFROM\s+STUDENTS\b"))

# 6. DELETE 기본 문법

기본 형식은 다음과 같습니다.

```sql
DELETE FROM 테이블명
WHERE 조건;
```

`DELETE FROM`은 **어느 테이블에서 삭제할지**,  
`WHERE`는 **어떤 행을 삭제할지** 지정합니다.

> `WHERE`가 빠지면 테이블 안의 모든 데이터가 삭제될 수 있습니다.

# 7. 삭제 전에는 SELECT로 먼저 확인하기

SQL-7 자료에서는 `홍길동` 학생을 삭제하기 전에 먼저 조회합니다.

## TODO 3
이름이 `홍길동`인 학생을 조회하세요.

In [ ]:
sql = '''
USE school;
SELECT * FROM students
WHERE TODO = 'TODO';
'''
_todo_id = 3
run_select(sql)

In [ ]:
# TODO 3 자동점검
check_select(3, sql,
             "USE school; SELECT * FROM students WHERE name='홍길동';",
             (r"\bWHERE\b", r"\bNAME\b"))

# 8. 이름을 조건으로 한 행 삭제하기

이제 방금 확인한 `홍길동` 학생을 삭제합니다.

## TODO 4
`WHERE name = '홍길동'` 조건을 사용하여 삭제하세요.

In [ ]:
sql = '''
USE school;
DELETE FROM students
WHERE TODO = 'TODO';
'''
_todo_id = 4
run_delete(sql)

In [ ]:
# TODO 4 자동점검
check_stage(4, "after_hong", sql,
            (r"\bDELETE\s+FROM\s+STUDENTS\b", r"\bWHERE\b", r"\bNAME\b"))

# 9. 삭제 후 다시 확인하기

삭제가 제대로 되었는지 다시 조회합니다.

## TODO 5
이름이 `홍길동`인 학생을 다시 조회하세요.  
정상이라면 결과 행이 없어야 합니다.

In [ ]:
sql = '''
USE school;
SELECT * FROM students
WHERE TODO = 'TODO';
'''
_todo_id = 5
run_select(sql)

In [ ]:
# TODO 5 자동점검
check_select(5, sql,
             "USE school; SELECT * FROM students WHERE name='홍길동';",
             (r"\bWHERE\b", r"\bNAME\b"))

# 10. id로 삭제하기

이름은 중복될 수 있지만 `id`는 각 행을 구분하는 고유값입니다.  
정확한 한 행을 삭제할 때는 `id`를 조건으로 사용하는 것이 안전합니다.

## TODO 6
먼저 `id = 5`인 학생을 조회하세요.

In [ ]:
sql = '''
USE school;
SELECT * FROM students
WHERE TODO = TODO;
'''
_todo_id = 6
run_select(sql)

In [ ]:
# TODO 6 자동점검
check_select(6, sql,
             "USE school; SELECT * FROM students WHERE id=5;",
             (r"\bWHERE\b", r"\bID\b"))

## TODO 7
`id = 5`인 학생을 삭제하세요.

In [ ]:
sql = '''
USE school;
DELETE FROM students
WHERE TODO = TODO;
'''
_todo_id = 7
run_delete(sql)

In [ ]:
# TODO 7 자동점검
check_stage(7, "after_id5", sql,
            (r"\bDELETE\s+FROM\s+STUDENTS\b", r"\bWHERE\b", r"\bID\b"))

# 11. 조건에 맞는 여러 데이터 삭제하기

`DELETE`는 조건에 맞는 행이 여러 개라면 여러 행을 한 번에 삭제합니다.

현재 `grade = 1`인 학생은 `혼이`, `김코딩` 두 명입니다.

## TODO 8
삭제 전에 `grade = 1`인 학생을 조회하세요.

In [ ]:
sql = '''
USE school;
SELECT * FROM students
WHERE TODO = TODO;
'''
_todo_id = 8
run_select(sql)

In [ ]:
# TODO 8 자동점검
check_select(8, sql,
             "USE school; SELECT * FROM students WHERE grade=1;",
             (r"\bWHERE\b", r"\bGRADE\b"))

## TODO 9
`grade = 1`인 학생을 모두 삭제하세요.

> 한 문장의 DELETE로 두 행이 삭제됩니다.

In [ ]:
sql = '''
USE school;
DELETE FROM students
WHERE TODO = TODO;
'''
_todo_id = 9
run_delete(sql)

In [ ]:
# TODO 9 자동점검
check_stage(9, "after_grade1", sql,
            (r"\bDELETE\s+FROM\s+STUDENTS\b", r"\bWHERE\b", r"\bGRADE\b"))

# 12. WHERE 없이 DELETE를 실행하면?

다음 SQL은 **실행하면 안 됩니다.**

```sql
DELETE FROM students;
```

`WHERE`가 없기 때문에 모든 행이 삭제될 수 있습니다.

아래에서는 실제 실행하지 않고 안전장치가 차단하는지만 확인합니다.

In [ ]:
danger_sql = '''
DELETE FROM students;
'''
explain_danger(danger_sql)

# 13. 안전한 DELETE 습관

삭제할 때는 다음 순서를 습관화합니다.

```text
1. SELECT로 삭제 대상 확인
2. WHERE 조건 다시 확인
3. DELETE 실행
4. 다시 SELECT로 확인
```

즉:

> **SELECT → DELETE → SELECT**

## TODO 10
`id = 7`인 데이터를 삭제하기 전에 먼저 조회하세요.

In [ ]:
sql = '''
USE school;
SELECT * FROM students
WHERE TODO = TODO;
'''
_todo_id = 10
run_select(sql)

In [ ]:
# TODO 10 자동점검
check_select(10, sql,
             "USE school; SELECT * FROM students WHERE id=7;",
             (r"\bWHERE\b", r"\bID\b"))

## TODO 11
`id = 7`인 데이터를 삭제하세요.

In [ ]:
sql = '''
USE school;
DELETE FROM students
WHERE TODO = TODO;
'''
_todo_id = 11
run_delete(sql)

In [ ]:
# TODO 11 자동점검
check_stage(11, "after_id7", sql,
            (r"\bDELETE\s+FROM\s+STUDENTS\b", r"\bWHERE\b", r"\bID\b"))

# 14. DELETE와 AUTO_INCREMENT

`DELETE`로 행을 지워도 `AUTO_INCREMENT` 번호는 일반적으로 이전 번호에서 이어집니다.

처음 데이터의 마지막 id는 `9`였습니다.  
중간 행들을 DELETE로 지운 뒤 새 데이터를 넣어 확인합니다.

## TODO 12
다음 데이터를 입력하세요.

- 이름: `새학생`
- 학년: `1`
- 학과: `컴퓨터과`

In [ ]:
sql = '''
USE school;
INSERT INTO students (name, grade, class_name)
VALUES ('TODO', TODO, 'TODO');
'''
_todo_id = 12
run_insert(sql)

In [ ]:
# TODO 12 자동점검
check_stage(12, "after_new", sql,
            (r"\bINSERT\s+INTO\s+STUDENTS\b", r"\bVALUES\b"))

## TODO 13
전체 데이터를 조회하여 `새학생`의 `id`를 확인하세요.

정상이라면 삭제된 번호를 다시 사용하는 것이 아니라 **id = 10**으로 이어집니다.

In [ ]:
sql = '''
USE school;
SELECT TODO FROM students;
'''
_todo_id = 13
run_select(sql)

In [ ]:
# TODO 13 자동점검
check_auto_increment(13)

# 15. TRUNCATE TABLE이란?

`TRUNCATE TABLE`은 테이블 구조는 그대로 두고 **테이블 안의 데이터를 한 번에 모두 비웁니다.**

```sql
TRUNCATE TABLE students;
```

### DELETE와 비교
- `DELETE` : WHERE 조건을 사용할 수 있음
- `TRUNCATE` : WHERE 조건을 사용하지 않음
- `DELETE` : AUTO_INCREMENT 값이 보통 이어짐
- `TRUNCATE` : AUTO_INCREMENT가 다시 시작되는 경우가 많음

> 이번 실습에서는 `students` 테이블에만 TRUNCATE를 허용합니다.

## TODO 14
`TRUNCATE TABLE`을 이용하여 `students` 테이블의 데이터를 모두 비우세요.

In [ ]:
sql = '''
USE school;
TODO TABLE TODO;
'''
_todo_id = 14
run_truncate(sql)

In [ ]:
# TODO 14 자동점검
check_empty(14, sql)

# 16. TRUNCATE 후 테이블 구조 확인하기

TRUNCATE는 데이터를 비우지만 테이블 자체를 삭제하지 않습니다.

## TODO 15
테이블 목록을 확인하여 `students` 테이블이 여전히 존재하는지 확인하세요.

In [ ]:
sql = '''
USE school;
SHOW TODO;
'''
_todo_id = 15
run_select(sql)

In [ ]:
# TODO 15 자동점검
check_table_exists(15)

# 17. TRUNCATE 후 AUTO_INCREMENT 확인하기

TRUNCATE 후 새 데이터를 넣으면 id가 다시 1부터 시작하는지 확인합니다.

## TODO 16
다음 데이터를 입력하세요.

- 이름: `다시시작`
- 학년: `1`
- 학과: `SQL반`

In [ ]:
sql = '''
USE school;
INSERT INTO students (name, grade, class_name)
VALUES ('TODO', TODO, 'TODO');
'''
_todo_id = 16
run_insert(sql)

In [ ]:
# TODO 16 자동점검
check_stage(16, "after_restart", sql,
            (r"\bINSERT\s+INTO\s+STUDENTS\b", r"\bVALUES\b"))

## TODO 17
전체 데이터를 조회하여 `다시시작`의 id를 확인하세요.

정상이라면 `id = 1`입니다.

In [ ]:
sql = '''
USE school;
SELECT TODO FROM TODO;
'''
_todo_id = 17
run_select(sql)

In [ ]:
# TODO 17 자동점검
check_select(17, sql,
             "USE school; SELECT * FROM students;",
             (r"\bSELECT\b", r"\bFROM\s+STUDENTS\b"))

# 18. DELETE와 TRUNCATE 차이 정리

| 구분 | DELETE | TRUNCATE |
|---|---|---|
| 특정 행 삭제 | 가능 | 불가능 |
| WHERE 사용 | 가능 | 불가능 |
| 전체 데이터 삭제 | 가능 | 가능 |
| 테이블 구조 유지 | 유지 | 유지 |
| AUTO_INCREMENT | 보통 이어짐 | 다시 시작되는 경우가 많음 |
| 주요 용도 | 조건에 맞는 행 삭제 | 테이블 데이터 전체 비우기 |

쉽게 기억하면:

> **DELETE = 서류철에서 특정 서류를 골라 버리기**  
> **TRUNCATE = 서류철 안의 서류를 한 번에 모두 비우기**

# 19. DROP TABLE과 혼동하지 않기

```sql
DROP TABLE students;
```

`DROP TABLE`은 데이터만 삭제하는 것이 아니라 **테이블 자체를 삭제**합니다.

따라서 이번 실습에서는 실제로 실행하지 않습니다.

```text
DELETE   → 데이터 삭제
TRUNCATE → 전체 데이터 비우기
DROP     → 테이블 자체 삭제
```

# 20. 자주 발생하는 실수

### ① WHERE 없이 DELETE 실행
```sql
DELETE FROM students;
```
→ 전체 데이터가 삭제될 수 있습니다.

### ② 삭제 전 SELECT를 하지 않음
먼저 같은 `WHERE` 조건으로 조회해 대상을 확인하세요.

### ③ 문자열에 따옴표를 쓰지 않음
잘못된 예:
```sql
DELETE FROM students WHERE name = 홍길동;
```
올바른 예:
```sql
DELETE FROM students WHERE name = '홍길동';
```

### ④ TRUNCATE에 WHERE 사용
잘못된 예:
```sql
TRUNCATE TABLE students WHERE grade = 3;
```
특정 조건만 삭제하려면 `DELETE ... WHERE ...`를 사용합니다.

### ⑤ DELETE와 DROP 혼동
`DROP TABLE students;`는 테이블 자체를 없애므로 실습 중에는 사용하지 않습니다.

# 21. 실습 데이터 복구

앞에서 TRUNCATE를 실행했으므로 다음 연습을 위해 SQL-7 시작 데이터 9개를 다시 복구합니다.

> 아래 셀은 수정하지 말고 실행하세요.

In [ ]:
restore_sql = '''
USE school;
TRUNCATE TABLE students;
INSERT INTO students (name, grade, class_name, created_at)
VALUES
('김부장', 3, '컴퓨터과', '2026-08-01 21:00:00'),
('혼이', 1, '영혼반', '2026-08-01 21:00:00'),
('라즈베리', 2, '임베디드반', '2026-08-01 21:00:00'),
('리눅스', 3, '서버반', '2026-08-01 21:00:00'),
('파이썬', 1, '프로그래밍반', '2026-08-01 21:00:00'),
('김코딩', 1, '컴퓨터과', '2026-08-01 21:00:00'),
('데이터왕', 2, '데이터베이스반', '2026-08-01 21:00:00'),
('홍길동', 2, '컴퓨터과', '2026-08-01 21:00:00'),
('이몽룡', 3, '데이터베이스반', '2026-08-01 21:00:00');
'''
ok,out,err=_mysql(restore_sql)
print("✅ SQL-7 시작 데이터 9개 복구 완료" if ok else err)

# 22. 종합 실습

## TODO 18
다음 조건을 모두 만족하세요.

1. 삭제 전에 `id = 3`인 학생을 확인한다.
2. `id = 3`인 학생을 `DELETE`로 삭제한다.
3. 다른 학생은 삭제하지 않는다.

아래 셀에는 **DELETE 문만** 작성하세요.  
바로 위 데이터 복구 셀을 먼저 실행해야 합니다.

In [ ]:
# 먼저 삭제 대상 확인
run_select("USE school; SELECT * FROM students WHERE id = 3;")

sql = '''
USE school;
DELETE FROM students
WHERE TODO = TODO;
'''
_todo_id = 18
run_delete(sql)

In [ ]:
# TODO 18 자동점검
expected = [r for r in STAGES["start"] if r[0] != '3']
actual = _snapshot()
syntax_ok, msg = _syntax_ok(sql, (r"\bDELETE\s+FROM\s+STUDENTS\b", r"\bWHERE\b", r"\bID\b"))
passed = syntax_ok and actual == expected
TODO_STATUS[18] = passed
print("🔎 TODO 18 자동점검")
print("✅ 통과: id를 사용해 정확히 한 행만 삭제했습니다." if passed else "❌ 재확인: id 조건과 삭제 결과를 확인하세요.")

# 23. 오늘 실습 코드 전체 정리

SQL-7의 핵심 흐름입니다.

```sql
USE school;

SELECT * FROM students;

-- 삭제 전 확인
SELECT * FROM students
WHERE name = '홍길동';

-- 특정 학생 삭제
DELETE FROM students
WHERE name = '홍길동';

-- id로 정확하게 삭제
SELECT * FROM students
WHERE id = 5;

DELETE FROM students
WHERE id = 5;

-- 조건에 맞는 여러 행 삭제
SELECT * FROM students
WHERE grade = 1;

DELETE FROM students
WHERE grade = 1;

-- 안전한 삭제 습관
SELECT * FROM students
WHERE id = 7;

DELETE FROM students
WHERE id = 7;

-- DELETE 후 AUTO_INCREMENT 확인
INSERT INTO students (name, grade, class_name)
VALUES ('새학생', 1, '컴퓨터과');

SELECT * FROM students;

-- 테이블 전체 데이터 비우기
TRUNCATE TABLE students;

SHOW TABLES;
SELECT * FROM students;

-- TRUNCATE 후 AUTO_INCREMENT 확인
INSERT INTO students (name, grade, class_name)
VALUES ('다시시작', 1, 'SQL반');

SELECT * FROM students;
```

> **안전한 삭제 습관:** `SELECT → DELETE → SELECT`

# 24. 전체 TODO 진행 현황

아래 셀에서 TODO 1~18의 자동점검 결과를 확인하세요.

- ✅ : 통과
- ❌ : 재확인 필요
- ⬜ : 아직 점검하지 않음

> 런타임을 다시 시작하면 진행 상태도 초기화됩니다.

In [ ]:
show_progress(18)

# 25. 오늘 배운 내용 정리

```text
DELETE FROM 테이블명 WHERE 조건;
```
→ 조건에 맞는 행 삭제

```text
TRUNCATE TABLE 테이블명;
```
→ 테이블 안의 데이터 전체 비우기

### 꼭 기억할 점
- DELETE 전에 반드시 SELECT로 대상을 확인한다.
- DELETE의 WHERE 조건을 특히 주의한다.
- 이름보다 id 같은 고유값이 더 안전할 수 있다.
- 조건에 맞는 행이 여러 개면 여러 행이 삭제된다.
- WHERE 없는 DELETE는 전체 데이터를 삭제할 수 있다.
- DELETE 후 AUTO_INCREMENT 번호는 보통 이어진다.
- TRUNCATE 후 AUTO_INCREMENT는 다시 시작되는 경우가 많다.
- TRUNCATE에는 WHERE를 사용할 수 없다.
- DROP TABLE은 테이블 자체를 삭제한다.

> **삭제는 확인이 생명입니다.**  
> **지우기 전에도 확인, 지운 뒤에도 확인합니다.**

# 26. 자기 점검

- [ ] DELETE의 역할을 설명할 수 있다.
- [ ] DELETE FROM과 WHERE를 사용할 수 있다.
- [ ] 삭제 전에 SELECT로 대상을 확인할 수 있다.
- [ ] id를 이용해 정확한 한 행을 삭제할 수 있다.
- [ ] 조건에 맞는 여러 행을 삭제할 수 있다.
- [ ] WHERE 없는 DELETE가 위험한 이유를 설명할 수 있다.
- [ ] DELETE 후 AUTO_INCREMENT가 이어질 수 있음을 안다.
- [ ] TRUNCATE TABLE을 사용할 수 있다.
- [ ] TRUNCATE에는 WHERE를 사용할 수 없음을 안다.
- [ ] TRUNCATE 후 AUTO_INCREMENT가 다시 시작되는 것을 확인할 수 있다.
- [ ] DELETE와 TRUNCATE의 차이를 설명할 수 있다.
- [ ] DROP TABLE은 테이블 자체를 삭제한다는 것을 안다.

# 27. [선택] 실습을 처음 상태로 되돌리기

실습 순서가 꼬였거나 SQL을 잘못 실행했다면 `RESET = True`로 바꾸어 실행하세요.

- SQL-7 시작 데이터 9개 복원
- AUTO_INCREMENT도 1부터 다시 시작
- TODO 진행 상태 초기화

In [ ]:
RESET = False

if RESET:
    reset_sql = '''
    USE school;
    TRUNCATE TABLE students;
    INSERT INTO students (name, grade, class_name, created_at)
    VALUES
    ('김부장', 3, '컴퓨터과', '2026-08-01 21:00:00'),
    ('혼이', 1, '영혼반', '2026-08-01 21:00:00'),
    ('라즈베리', 2, '임베디드반', '2026-08-01 21:00:00'),
    ('리눅스', 3, '서버반', '2026-08-01 21:00:00'),
    ('파이썬', 1, '프로그래밍반', '2026-08-01 21:00:00'),
    ('김코딩', 1, '컴퓨터과', '2026-08-01 21:00:00'),
    ('데이터왕', 2, '데이터베이스반', '2026-08-01 21:00:00'),
    ('홍길동', 2, '컴퓨터과', '2026-08-01 21:00:00'),
    ('이몽룡', 3, '데이터베이스반', '2026-08-01 21:00:00');
    '''
    ok,out,err=_mysql(reset_sql)
    if ok:
        TODO_STATUS.clear()
        print("✅ SQL-7 시작 상태로 복원했습니다.")
        print("✅ TODO 진행 상태도 초기화했습니다.")
    else:
        print("❌ 초기화 오류")
        print(err)
else:
    print("ℹ️ RESET = False 입니다. 데이터는 변경하지 않았습니다.")

# 다음 시간 예고

## [SQL 실습 #08] MySQL 정렬과 개수 제한 더 깊게 보기
### ORDER BY, LIMIT, OFFSET

다음 시간에는 데이터를 원하는 순서로 정렬하고 필요한 개수만 조회하는 방법을 연습합니다.